<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro to Reinforcement Learning

In [261]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Step:
    action: Any
    prob: float  # New: the probability that we ended up here
    reward: float
    state: Any
class MDP:
    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> list[Step]:
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        raise NotImplementedError
    def discount(self) -> float:
        raise NotImplementedError
class FlakyTramMDP(MDP):
    def __init__(self, num_locs: int, failure_prob: float):
        self.num_locs = num_locs
        self.failure_prob = failure_prob
    def start_state(self) -> Any:
        return 1

    def successors(self, state: Any) -> list[Step]:
        successors = []
        # Walk
        if state + 1 <= self.num_locs:
            successors.append(Step(action="walk", prob=1, reward=-1, state=state + 1))
        # Tram
        if 2 * state <= self.num_locs:
            # Success: move to desired state
            successors.append(Step(action="tram", prob=1 - self.failure_prob, reward=-2, state=2 * state))
            # Failure: stay in the same state
            successors.append(Step(action="tram", prob=self.failure_prob, reward=-2, state=state))
        return successors
    def is_end(self, state: Any) -> bool:
        return state == self.num_locs
    def discount(self) -> float:
        # No discounting for now
        return 1

#### Environment

In [262]:
import numpy as np

mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
np.random.seed(1)

#### Agent

In [263]:
def tram_if_possible_policy(mdp: MDP, state: int) -> str:
    """Need the MDP to know the number of locations to make sure we can take the tram."""
    if state * 2 <= mdp.num_locs:
        return "tram"
    else:
        return "walk"

In [264]:
from typing import Callable
Policy = Callable[[Any], Any]

class RLAlgorithm:
    """
    Abstract class for an RL algorithm, which does two things:
    1. Sends actions to the environment
    2. Incorporates feedback from the environment
    """
    def get_action(self, state: Any) -> Any:
        raise NotImplementedError
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        raise NotImplementedError

class StaticAgent(RLAlgorithm):
    def __init__(self, policy: Policy):
        self.policy = policy
    def get_action(self, state: Any) -> Any:
        return self.policy(state)
    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        # Setup Reward Feedback Loigic
        pass

In [265]:
from functools import partial

policy = partial(tram_if_possible_policy, mdp)
rl = StaticAgent(policy=policy)
rl.get_action(state=1)

print(f"Action taken: {action}")

Action taken: tram


#### Rollout

In [266]:
def sample_transition(mdp: MDP, state: Any, action: Any) -> Step:
    """
    Samples a transition from the MDP: samples s' with probability T(s, a, s').
    Returns:
    - reward: the reward for the transition
    - next_state: the next state
    - is_end: whether the next state is an end state
    """
    # Get successors given (state, action)
    successors = [successor for successor in mdp.successors(state) if successor.action == action]
    if len(successors) == 0:
        raise ValueError(f"No successors found for state {state} and action {action}")
    # Sample a successor based on its probabilities
    probs = [successor.prob for successor in successors]
    choice = np.random.choice(len(successors), p=probs)
    step = successors[choice]
    return step

In [267]:
def compute_utility(steps: list[Step], discount: float) -> float:
    """Computes the utility (discounted sum of rewards) of a rollout."""
    rewards = [step.reward * discount ** i for i, step in enumerate(steps)]
    utility = sum(rewards)
    return utility

In [268]:
class Rollout:
    """Represents a rollout of an MDP (sequence of actions that produces a utility)."""
    steps: list[Step]
    discount: float
    utility: float  # Discounted sum of rewards
    def __init__(self, steps: list[Step], discount: float):
        self.steps = steps
        self.discount = discount
        self.utility = compute_utility(steps, discount)

In [269]:
def simulate(mdp: MDP, rl: RLAlgorithm, num_trials: int) -> float:
    """
    Runs the RL algorithm on the MDP.
    Return the mean utility of the rollouts.
    """
    utilities = []
    # Repeat multiple times
    for trial in range(num_trials):
        # Environment state
        state = mdp.start_state()
        steps = []
        while not mdp.is_end(state):
            # Agent sends action to environment
            action = rl.get_action(state)
            # Environment sends reward and next state to agent
            step = sample_transition(mdp, state, action)
            rl.incorporate_feedback(state, action, step.reward, step.state, mdp.is_end(step.state))
            steps.append(step)
            state = step.state
        # Compute utility of this rollout
        rollout = Rollout(steps=steps, discount=mdp.discount())
        utilities.append(rollout.utility)
    mean_utility = np.mean(utilities).item()
    return mean_utility

In [270]:
LEADERBOARD = {}  # method -> value

def update_leaderboard(method: str, value: float) -> dict[str, float]:
    LEADERBOARD[method] = value
    return LEADERBOARD

In [271]:
value = simulate(mdp, rl, num_trials=10)
print(f"Value from Rollout: {value}")

leaderboard = update_leaderboard("tram_if_possible_policy", value)

print(leaderboard)

Value from Rollout: -11.6
{'tram_if_possible_policy': -11.6}


# Model-based RL

In [272]:
# Tram 탈 수 있으면 확률적으로 tram을 타고, 그 외에는 walk를 항상 선택하는 policy

def walk_tram_policy(num_locs: int, state: Any) -> Any:
    """Chooses a random valid action."""
    if 2 * state <= num_locs:
        # Can take the tram
        return np.random.choice(["walk", "tram"]).item()
    else:
        # Can only walk
        return "walk"

In [273]:
def try_out_exploration_policy(exploration_policy: Policy):
    state=1
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")
    action = exploration_policy(state)
    print(f"Action taken: {action}, at State {state}")

    state=6
    action = exploration_policy(state=6)
    print(f"Action taken: {action}, at State {state}")


In [274]:
exploration_policy = partial(walk_tram_policy, mdp.num_locs)
try_out_exploration_policy(exploration_policy)

Action taken: walk, at State 1
Action taken: tram, at State 1
Action taken: tram, at State 1
Action taken: walk, at State 6


#### Exploration으로 MDP를 재구축 하자

In [275]:
from collections import defaultdict
import numpy as np
from typing import Any, List, Dict, Tuple

class Step:
    def __init__(self, action: Any, prob: float, reward: float, state: Any):
        self.action = action
        self.prob = prob
        self.reward = reward
        self.state = state

    def __repr__(self):
        return f"Step(action={self.action}, prob={self.prob}, reward={self.reward}, state={self.state})"

class EstimatedMDP:
    """An MDP whose start state, rewards, transitions, and is_end are learned from feedback during RL."""
    def __init__(self, discount: float):
        self.start_state_ = None
        self.rewards = defaultdict(float)  # (state, action, state') -> reward
        self.transitions = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))  # state -> action -> state' -> count
        self.end_states = set()
        self.discount_ = discount

    def start_state(self) -> Any:
        return self.start_state_

    def successors(self, state: Any) -> List[Step]:
        """Compute successors based on the transition counts and rewards."""
        successors = []
        for action, next_state_counts in self.transitions[state].items():
            total_count = sum(next_state_counts.values())
            for next_state, count in next_state_counts.items():
                prob = count / total_count
                reward = self.rewards[(state, action, next_state)]
                step = Step(action=action, prob=prob, reward=reward, state=next_state)
                successors.append(step)
        return successors

    def is_end(self, state: Any) -> bool:
        return state in self.end_states

    def discount(self) -> float:
        return self.discount_

    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        if self.start_state_ is None:
            self.start_state_ = state
        self.rewards[(state, action, next_state)] = reward
        self.transitions[state][action][next_state] += 1
        if is_end:
            self.end_states.add(next_state)

    def asdict(self) -> Dict[str, Any]:
        return {
            "start_state_": self.start_state_,
            "rewards": self.rewards,
            "transition_counts": self.transitions,
            "end_states": list(self.end_states),
        }

def get_states(mdp: EstimatedMDP) -> List[Any]:
    states = set()
    def recurse(state: Any):
        if state in states: return
        states.add(state)
        for step in mdp.successors(state):
            recurse(step.state)
    if mdp.start_state() is not None: recurse(mdp.start_state())
    return list(states)

def get_initial_values(mdp: EstimatedMDP) -> Dict[Any, float]:
    states = get_states(mdp)
    return {state: 0 if mdp.is_end(state) else -100 for state in states}

def get_action_successors(mdp: EstimatedMDP, state: Any) -> Dict[str, List[Step]]:
    successors = mdp.successors(state)
    action_to_successors = defaultdict(list)
    for step in successors:
        action_to_successors[step.action].append(step)
    return action_to_successors

def compute_q_value(successors: List[Step], discount: float, values: Dict[Any, float]) -> float:
    return sum(step.prob * (step.reward + discount * values.get(step.state, -100)) for step in successors)


def compute_distance(values: dict[Any, float], new_values: dict[Any, float]) -> float:
    """Compute the distance between two sets of values."""
    distances = [abs(values[state] - new_values[state]) for state in values]
    max_distance = max(distances)
    return max_distance

@dataclass(frozen=True)
class ValueIterationResult:
    values: dict[Any, float]  # state -> value of that state
    pi: dict[Any, Any]  # state -> action for that state
    distances: list[float]  # iteration -> change in value that iteration
def value_iteration(mdp: MDP, max_iters: int = 100, tolerance: float = 1e-5) -> ValueIterationResult:
    """
    Compute the value of the optimal policy.
    Returns:
    - values: dict[Any, float]: state -> optimal value of that state
    - pi: dict[Any, Any]: state -> optimal action for that state
    - distances: list[float]: distance between values and new values at each iteration
    """
    # Initialize values of each state
    values = get_initial_values(mdp)
    # Initialize the policy (state -> action)
    pi = {state: None for state in values}
    distances = []
    for iter in range(max_iters):
        new_values = {}  # state -> V_iter(state)
        for state in values:
            if mdp.is_end(state):
                new_values[state] = 0
                continue
            new_values[state], pi[state] = value_iteration_for_state(mdp, state, values)
        # Check for convergence
        distance = compute_distance(values, new_values)
        distances.append(distance)
        if distance < tolerance:
            break
        # Update values to the next iteration
        values = new_values
    return ValueIterationResult(values=values, pi=pi, distances=distances)
def value_iteration_for_state(mdp: MDP, state: Any, values: dict[Any, float]) -> tuple[float, Any]:
    """Compute optimal value V`*`(state) and optimal policy π`*`(state)."""
    # V^* = max_a Q(s, a, V^*)
    # For each action, compute its Q-value
    actions = []
    q_values = []
    # Look at all actions from state
    for action, successors in get_action_successors(mdp, state).items():
        actions.append(action)
        q_values.append(compute_q_value(successors, mdp.discount(), values))
    # Take the best action
    value = np.max(q_values)
    action = actions[np.argmax(q_values)]
    return value, action

class ModelBasedValueIteration:
    def __init__(self, exploration_policy: Any, discount: float):
        self.exploration_policy = exploration_policy
        self.exploitation_policy = None
        self.mdp = EstimatedMDP(discount=discount)

    def run_value_iteration(self):
        result = value_iteration(self.mdp)
        self.exploitation_policy = result.pi

    def get_action(self, state: Any) -> Any:
        if self.exploitation_policy is None or state not in self.exploitation_policy: return self.exploration_policy(state)
        return self.exploitation_policy[state]

    def incorporate_feedback(self, state, action, reward, next_state, is_end):
        self.mdp.incorporate_feedback(state, action, reward, next_state, is_end)

In [276]:
def compare_mdps(true_mdp: MDP, estimated_mdp: MDP):
    print(f"True MDP Start State: {true_mdp.start_state()}")
    print(f"Estimated MDP Start State: {estimated_mdp.start_state()}")
    print(f"True Successors: {true_mdp.successors(state=1)}")
    print(f"Estimated Successors: {estimated_mdp.successors(state=1)}")
    print(f"True End State (is_end(state=9)?: {true_mdp.is_end(state=9)}")
    print(f"Estimated End State (is_end(state=9)?: {estimated_mdp.is_end(state=9)}")

In [277]:
rl = ModelBasedValueIteration(exploration_policy=exploration_policy, discount=1)

value = simulate(mdp, rl, num_trials=10)
print(f"Average Value from Rollout: {value}")
leaderboard = update_leaderboard("model_based_value_iteration.explore", value)
print(f"Update to Leaderboard: {leaderboard}")

print("\n---------- Compare True MDP vs. Estimated MDP ----------")
compare_mdps(mdp, rl.mdp)


Average Value from Rollout: -10.7
Update to Leaderboard: {'tram_if_possible_policy': -11.6, 'model_based_value_iteration.explore': -10.7}

---------- Compare True MDP vs. Estimated MDP ----------
True MDP Start State: 1
Estimated MDP Start State: 1
True Successors: [Step(action=walk, prob=1, reward=-1, state=2), Step(action=tram, prob=0.6, reward=-2, state=2), Step(action=tram, prob=0.4, reward=-2, state=1)]
Estimated Successors: [Step(action=tram, prob=0.4444444444444444, reward=-2, state=1), Step(action=tram, prob=0.5555555555555556, reward=-2, state=2), Step(action=walk, prob=1.0, reward=-1, state=2)]
True End State (is_end(state=9)?: False
Estimated End State (is_end(state=9)?: False


#### Value iteration을 실행하자

In [278]:
def compare_policies(true_policy: Dict[Any, Any], estimated_policy: Dict[Any, Any]):
    """Compares the policies from the true MDP and the estimated MDP."""
    print("----------- Comparing Policies -----------")
    print("True Policy:")
    for state in sorted(true_policy.keys()):
        print(f"State {state}: Action {true_policy[state]}")

    print("\nEstimated Policy:")
    for state in sorted(estimated_policy.keys()):
        action = estimated_policy.get(state, None)
        print(f"State {state}: Action {action if action is not None else 'No action defined'}")

# Example usage of compare_policies
rl.run_value_iteration()  # Ensure value iteration has been run to populate the exploitation policy

true_policy = value_iteration(mdp).pi  # Obtain the policy from the true MDP
estimated_policy = rl.exploitation_policy  # The policy from the estimated MDP

compare_policies(true_policy, estimated_policy)


----------- Comparing Policies -----------
True Policy:
State 1: Action walk
State 2: Action walk
State 3: Action walk
State 4: Action walk
State 5: Action tram
State 6: Action walk
State 7: Action walk
State 8: Action walk
State 9: Action walk
State 10: Action None

Estimated Policy:
State 1: Action walk
State 2: Action walk
State 3: Action walk
State 4: Action tram
State 5: Action walk
State 6: Action walk
State 7: Action walk
State 8: Action walk
State 9: Action walk
State 10: Action No action defined


#### Model-free RL

In [279]:
class ModelFreeMonteCarlo(RLAlgorithm):
    def __init__(self, exploration_policy: Policy, epsilon: float, discount: float):
        self.exploration_policy = exploration_policy
        self.epsilon = epsilon
        self.discount = discount
        self.sum_utilities = defaultdict(lambda: defaultdict(float))  # state -> action -> sum of utility from (state, action)
        self.counts = defaultdict(lambda: defaultdict(int))  # state -> action -> visitation count
        self.start_state = None
        self.rollout: list[Step] = []

    def get_action(self, state: Any) -> Any:
        if len(self.counts[state]) == 0:
            # If no actions have been tried yet, choose a random action
            action = self.exploration_policy(state)
            print(f"State {state}: Choosing random action {action} (initial exploration)")
            return action

        # Do epsilon-greedy
        if np.random.random() < self.epsilon:
            action = self.exploration_policy(state)
            print(f"State {state}: Choosing random action {action} (epsilon-greedy)")
            return action
        else:
            action = self.pi(state)
            print(f"State {state}: Choosing best action {action} (based on Q-values)")
            return action

    def pi(self, state: Any) -> Any:
        """Return the policy corresponding to the current Q-values."""
        actions = list(self.counts[state].keys())
        q_values = [self.Q(state, action) for action in actions]
        action = actions[np.argmax(q_values).item()]
        return action

    def Q(self, state: Any, action: Any) -> float:
        """Compute the estimated Q-values Q(state, action) using the running sums and counts."""
        sum_utility = self.sum_utilities[state][action]
        count = self.counts[state][action]
        value = sum_utility / count if count > 0 else 0.0
        return value

    def incorporate_feedback(self, state: Any, action: Any, reward: Any, next_state: Any, is_end: bool) -> None:
        print(f"Incorporating feedback: state={state}, action={action}, reward={reward}, next_state={next_state}, is_end={is_end}")

        # Add this piece of feedback (state, action, reward, next_state) to the history
        if self.start_state is None:
            self.start_state = state

        self.rollout.append(Step(action=action, prob=1, reward=reward, state=next_state))

        # At the end of the episode, update the statistics needed for computing Q-values
        if is_end:
            print("End of episode reached.")
            utilities = [0] * (len(self.rollout) + 1)
            # Walk backwards and compute the utilities for each step
            for i, step in reversed(list(enumerate(self.rollout))):
                state = self.start_state if i == 0 else self.rollout[i - 1].state
                utilities[i] = step.reward + self.discount * utilities[i + 1]

                # Update the running sums
                self.sum_utilities[state][step.action] += utilities[i]
                self.counts[state][step.action] += 1

                print(f"Step {i}: Updated utility for (state={state}, action={step.action}) to {self.sum_utilities[state][step.action]}. Count is now {self.counts[state][step.action]}.")

            # Display the updated Q-values
            print("Updated Q-values after episode:")
            for s in self.sum_utilities.keys():
                for a in self.sum_utilities[s].keys():
                    q_value = self.Q(s, a)
                    print(f"Q({s}, {a}) = {q_value} (sum_utilities={self.sum_utilities[s][a]}, count={self.counts[s][a]})")

            # Reset history
            self.start_state = None
            self.rollout = []

In [280]:
exploration_policy = partial(walk_tram_policy, mdp.num_locs)
rl = ModelFreeMonteCarlo(exploration_policy=exploration_policy, epsilon=0.4, discount=1)

In [281]:
value = simulate(mdp, rl, num_trials=20)
leaderboard = update_leaderboard("model_free_monte_carlo", value)

State 1: Choosing random action tram (initial exploration)
Incorporating feedback: state=1, action=tram, reward=-2, next_state=2, is_end=False
State 2: Choosing random action walk (initial exploration)
Incorporating feedback: state=2, action=walk, reward=-1, next_state=3, is_end=False
State 3: Choosing random action walk (initial exploration)
Incorporating feedback: state=3, action=walk, reward=-1, next_state=4, is_end=False
State 4: Choosing random action tram (initial exploration)
Incorporating feedback: state=4, action=tram, reward=-2, next_state=8, is_end=False
State 8: Choosing random action walk (initial exploration)
Incorporating feedback: state=8, action=walk, reward=-1, next_state=9, is_end=False
State 9: Choosing random action walk (initial exploration)
Incorporating feedback: state=9, action=walk, reward=-1, next_state=10, is_end=True
End of episode reached.
Step 5: Updated utility for (state=9, action=walk) to -1.0. Count is now 1.
Step 4: Updated utility for (state=8, acti

#### SARSA

In [282]:
class SARSA(RLAlgorithm):
    def __init__(
        self,
        exploration_policy: Policy,
        epsilon: float,
        discount: float,
        learning_rate: float
    ):
        self.exploration_policy = exploration_policy
        self.epsilon = epsilon
        self.discount = discount
        self.learning_rate = learning_rate

        # state -> action -> Q-value
        self.Q = defaultdict(lambda: defaultdict(float))

    def get_action(self, state: Any) -> Any:

        # No prior knowledge for this state
        if len(self.Q[state]) == 0:
            action = self.exploration_policy(state)

            print(
                f"State {state}: Choosing random action {action} "
                f"(initial exploration - no known Q-values)"
            )

            return action

        # Epsilon-greedy exploration
        if np.random.random() < self.epsilon:

            action = self.exploration_policy(state)

            print(
                f"State {state}: Choosing random action {action} "
                f"(epsilon-greedy exploration, epsilon={self.epsilon})"
            )

            return action

        # Exploitation
        else:

            action = self.pi(state)

            print(
                f"State {state}: Choosing best action {action} "
                f"(exploitation from current Q-values)"
            )

            return action

    def pi(self, state: Any) -> Any:
        """Return the greedy policy corresponding to current Q-values."""

        actions = list(self.Q[state].keys())

        if len(actions) == 0:
            print(f"State {state}: No available actions in Q-table yet.")
            return None

        q_values = [self.Q[state][action] for action in actions]

        print(f"\n[Policy Evaluation for State {state}]")
        for action, q_value in zip(actions, q_values):
            print(f"  Q({state}, {action}) = {q_value:.4f}")

        best_action = actions[np.argmax(q_values).item()]

        print(f"  -> Best action selected: {best_action}")

        return best_action

    def incorporate_feedback(
        self,
        state: Any,
        action: Any,
        reward: Any,
        next_state: Any,
        is_end: bool
    ) -> None:

        print("\n" + "=" * 70)
        print("[SARSA Feedback Incorporation]")
        print(
            f"Observed transition:\n"
            f"  state={state}\n"
            f"  action={action}\n"
            f"  reward={reward}\n"
            f"  next_state={next_state}\n"
            f"  is_end={is_end}"
        )

        # Current Q before update
        old_q = self.Q[state][action]

        print(f"\nCurrent Q-value:")
        print(f"  Q({state}, {action}) = {old_q:.4f}")

        # Terminal state case
        if is_end:

            next_action = None
            next_q = 0.0

            print("\nTerminal state reached.")
            print("  No next action.")
            print("  Future utility = 0")

        else:

            # SARSA uses ON-POLICY next action
            next_action = self.get_action(next_state)

            next_q = self.Q[next_state].get(next_action, 0.0)

            print("\n[On-Policy Next Step]")
            print(f"  Next action chosen by current policy: {next_action}")
            print(f"  Q({next_state}, {next_action}) = {next_q:.4f}")

        # SARSA target:
        # target = r + gamma * Q(s', a')
        utility = reward + self.discount * next_q

        print("\n[SARSA Target Calculation]")
        print(
            f"  utility = reward + discount * next_Q\n"
            f"          = {reward} + ({self.discount} * {next_q:.4f})\n"
            f"          = {utility:.4f}"
        )

        # TD Error
        td_error = utility - old_q

        print("\n[TD Error]")
        print(
            f"  TD Error = target - current_Q\n"
            f"           = {utility:.4f} - {old_q:.4f}\n"
            f"           = {td_error:.4f}"
        )

        # Update rule:
        # Q(s,a) <- Q(s,a) + alpha * TD_error
        update_amount = self.learning_rate * td_error

        self.Q[state][action] += update_amount

        new_q = self.Q[state][action]

        print("\n[Q-value Update]")
        print(
            f"  Q({state}, {action}) ← {old_q:.4f} + "
            f"{self.learning_rate} * ({td_error:.4f})"
        )

        print(f"  Update amount = {update_amount:.4f}")
        print(f"  New Q({state}, {action}) = {new_q:.4f}")

        # Full Q-table snapshot
        print("\n[Updated Q-table Snapshot]")
        for s in self.Q.keys():
            for a in self.Q[s].keys():
                print(f"  Q({s}, {a}) = {self.Q[s][a]:.4f}")

        print("=" * 70)

In [283]:
exploration_policy = partial(walk_tram_policy, mdp.num_locs)
rl = SARSA(exploration_policy=exploration_policy, epsilon=0.4, discount=1, learning_rate=0.1)

In [284]:
value = simulate(mdp, rl, num_trials=20)
leaderboard = update_leaderboard("sarsa", value)

State 1: Choosing random action tram (initial exploration - no known Q-values)

[SARSA Feedback Incorporation]
Observed transition:
  state=1
  action=tram
  reward=-2
  next_state=2
  is_end=False

Current Q-value:
  Q(1, tram) = 0.0000
State 2: Choosing random action walk (initial exploration - no known Q-values)

[On-Policy Next Step]
  Next action chosen by current policy: walk
  Q(2, walk) = 0.0000

[SARSA Target Calculation]
  utility = reward + discount * next_Q
          = -2 + (1 * 0.0000)
          = -2.0000

[TD Error]
  TD Error = target - current_Q
           = -2.0000 - 0.0000
           = -2.0000

[Q-value Update]
  Q(1, tram) ← 0.0000 + 0.1 * (-2.0000)
  Update amount = -0.2000
  New Q(1, tram) = -0.2000

[Updated Q-table Snapshot]
  Q(1, tram) = -0.2000
State 2: Choosing random action walk (initial exploration - no known Q-values)

[SARSA Feedback Incorporation]
Observed transition:
  state=2
  action=walk
  reward=-1
  next_state=3
  is_end=False

Current Q-value:
  Q

#### Q-Learning

In [285]:
class QLearning(SARSA):
    """Q-learning is SARSA, but with an off-policy greedy target policy."""

    def incorporate_feedback(
        self,
        state: Any,
        action: Any,
        reward: Any,
        next_state: Any,
        is_end: bool
    ) -> None:

        print("\n" + "=" * 70)
        print("[Q-Learning Feedback Incorporation]")
        print(
            f"Observed transition:\n"
            f"  state={state}\n"
            f"  action={action}\n"
            f"  reward={reward}\n"
            f"  next_state={next_state}\n"
            f"  is_end={is_end}"
        )

        # Current Q before update
        old_q = self.Q[state][action]

        print(f"\nCurrent Q-value:")
        print(f"  Q({state}, {action}) = {old_q:.4f}")

        # Terminal state
        if is_end:

            next_action = None
            next_q = 0.0

            print("\nTerminal state reached.")
            print("  No future action needed.")
            print("  Future utility = 0")

        else:

            print("\n[Off-Policy Greedy Evaluation]")
            print(
                "  Q-Learning ignores exploration policy here.\n"
                "  It evaluates the GREEDY best next action using pi(next_state)."
            )

            # OFF-POLICY:
            # Use greedy policy, not exploratory get_action
            next_action = self.pi(next_state)

            if next_action is None:
                next_q = 0.0

                print(
                    f"  No known actions yet for next_state={next_state}.\n"
                    f"  Default next Q = 0"
                )

            else:
                next_q = self.Q[next_state].get(next_action, 0.0)

                print(f"  Best greedy next action: {next_action}")
                print(f"  Q({next_state}, {next_action}) = {next_q:.4f}")

        # Q-Learning target:
        # target = r + gamma * max_a' Q(s', a')
        utility = reward + self.discount * next_q

        print("\n[Q-Learning Target Calculation]")
        print(
            f"  utility = reward + discount * max_next_Q\n"
            f"          = {reward} + ({self.discount} * {next_q:.4f})\n"
            f"          = {utility:.4f}"
        )

        # TD Error
        td_error = utility - old_q

        print("\n[TD Error]")
        print(
            f"  TD Error = target - current_Q\n"
            f"           = {utility:.4f} - {old_q:.4f}\n"
            f"           = {td_error:.4f}"
        )

        # Update:
        # Q(s,a) <- Q(s,a) + alpha * TD_error
        update_amount = self.learning_rate * td_error

        self.Q[state][action] += update_amount

        new_q = self.Q[state][action]

        print("\n[Q-value Update]")
        print(
            f"  Q({state}, {action}) ← {old_q:.4f} + "
            f"{self.learning_rate} * ({td_error:.4f})"
        )

        print(f"  Update amount = {update_amount:.4f}")
        print(f"  New Q({state}, {action}) = {new_q:.4f}")

        # Compare SARSA vs Q-Learning conceptually
        print("\n[Key Q-Learning Characteristic]")
        print(
            "  Target policy: GREEDY (max Q)\n"
            "  Behavior policy: May still explore externally\n"
            "  => OFF-POLICY learning"
        )

        # Full Q-table snapshot
        print("\n[Updated Q-table Snapshot]")
        for s in self.Q.keys():
            for a in self.Q[s].keys():
                print(f"  Q({s}, {a}) = {self.Q[s][a]:.4f}")

        print("=" * 70)

In [288]:
rl = QLearning(exploration_policy=exploration_policy, epsilon=0.4, discount=1, learning_rate=0.1)


In [290]:
value = simulate(mdp, rl, num_trials=10)
leaderboard = update_leaderboard("q_learning", value)


[Policy Evaluation for State 1]
  Q(1, tram) = -0.5630
  Q(1, walk) = -0.3442
  -> Best action selected: walk
State 1: Choosing best action walk (exploitation from current Q-values)

[Q-Learning Feedback Incorporation]
Observed transition:
  state=1
  action=walk
  reward=-1
  next_state=2
  is_end=False

Current Q-value:
  Q(1, walk) = -0.3442

[Off-Policy Greedy Evaluation]
  Q-Learning ignores exploration policy here.
  It evaluates the GREEDY best next action using pi(next_state).

[Policy Evaluation for State 2]
  Q(2, walk) = -0.4000
  Q(2, tram) = -0.4000
  -> Best action selected: walk
  Best greedy next action: walk
  Q(2, walk) = -0.4000

[Q-Learning Target Calculation]
  utility = reward + discount * max_next_Q
          = -1 + (1 * -0.4000)
          = -1.4000

[TD Error]
  TD Error = target - current_Q
           = -1.4000 - -0.3442
           = -1.0558

[Q-value Update]
  Q(1, walk) ← -0.3442 + 0.1 * (-1.0558)
  Update amount = -0.1056
  New Q(1, walk) = -0.4498

[Key Q-